# EduHarness — Colab API Setup（混合路线）

1. 本地：`bash scripts/pack_colab_src.sh` → 上传 `MyDrive/eduharness/eduharness-colab.zip`
2. Colab 🔑 **Secrets** 添加名为 **`openai`** 的 key，并开启 **Notebook access**
3. 运行本 notebook；runs 在 `MyDrive/eduharness/runs/`

> 每个 notebook 独立内核；`02`/`03` 会通过 `ensure_api_key()` 自动读取 Secret `openai`。


In [3]:
from __future__ import annotations
import json, os, sys, subprocess
from getpass import getpass
from pathlib import Path

from google.colab import drive  # type: ignore
drive.mount("/content/drive", force_remount=False)

DRIVE_ZIP = Path("/content/drive/MyDrive/eduharness/eduharness-colab.zip")
REPO = Path("/content")
RUN_DIR = Path("/content/drive/MyDrive/eduharness/runs")
REQUEST_REL = "examples/sample_request.json"

if not DRIVE_ZIP.is_file():
    raise FileNotFoundError(f"请先把 zip 上传到 Drive: {DRIVE_ZIP}")

get_ipython().system(f'unzip -o "{DRIVE_ZIP}" -d /content')

if not (REPO / "eduharness" / "pipeline.py").is_file():
    raise FileNotFoundError(f"解压后未找到 {REPO}/eduharness/pipeline.py")

RUN_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

os.environ["EDUHARNESS_TEXT_MODEL"] = "gpt-5"
os.environ["EDUHARNESS_VISION_MODEL"] = "gpt-5"
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")

# 推荐：Colab 左侧 🔑 Secrets 添加 OPENAI_API_KEY，后续 notebook 会自动读取
from eduharness.colab_session import ensure_api_key
ensure_api_key()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])

cfg = {
    "REPO": str(REPO),
    "RUN_DIR": str(RUN_DIR),
    "REQUEST": str(REPO / REQUEST_REL),
}
Path("/content/eduharness_colab_env.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
print(json.dumps(cfg, indent=2, ensure_ascii=False))
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/MyDrive/eduharness/eduharness-colab.zip
  inflating: /content/eduharness/config.py  
  inflating: /content/eduharness/colab_session.py  
 extracting: /content/eduharness/stage1/__init__.py  
  inflating: /content/eduharness/stage1/main_agent.py  
  inflating: /content/eduharness/providers/__init__.py  
  inflating: /content/eduharness/providers/openai_provider.py  
  inflating: /content/eduharness/providers/base.py  
  inflating: /content/eduharness/__init__.py  
  inflating: /content/eduharness/pipeline.py  
  inflating: /content/eduharness/templates/registry.py  
 extracting: /content/eduharness/templates/__init__.py  
  inflating: /content/eduharness/templates/schemas/fill_blank.json  
  inflating: /content/eduharness/templates/schemas/multiple_choice.json  
  inflating: /content/eduharness/templates/schemas/physics_lever.json  
  

In [4]:
# 2) 本地衔接（仅提示）
print("""
推荐一键 API 流程：上传并打开 notebooks/06_colab_api_oneshot.ipynb
  PHASE='bootstrap' → 本地 2-render → PHASE='review' → 本地 stage 3

分步（旧）：02_stage1_plan → 03_stage2_render (MODE='prepare')

同步 run：MyDrive/eduharness/runs/<request_id> ↔ 本地 runs/<request_id>
""")



下一步：02_stage1_plan.ipynb → 03_stage2_render.ipynb (MODE='prepare')

同步 run：MyDrive/eduharness/runs/lever_physics_01 ↔ 本地 runs/lever_physics_01

本地渲染：
  cd ~/Desktop/eduharness && source .venv/bin/activate
  python -m eduharness \
    --request-json examples/sample_request.json \
    --run-dir runs/lever_physics_01 \
    --stage 2-render

